# 评估器诊断（探索性，不是核对本身）

D10 核对第二次还差 +0.0333。AP 计算、配对、NMS 已经在本地跟 Ultralytics 8.4.182 的代码逐个对过，完全一致。
剩下可能的差别只有两类：**图片预处理**（我们照服务器用 PIL 缩放，Ultralytics 用 OpenCV INTER_LINEAR）和 **标注/图片的读取规则**（Ultralytics 会去掉重复框、跳过坐标越界的图）。

这个 notebook 在 merged_v5 val 上把这几样一样一样换掉，看差距来自哪里。CPU 就够，大约 15–20 分钟。跑完把最后一格的输出整段贴给 Claude。

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/mixed_dataset"
REPO_URL = "https://github.com/xiaolin200206/Snapdurian_deploy.git"
ULTRALYTICS = "8.4.182"
ONNX_NAME = "durian_11cls_640_v5.onnx"

In [ ]:
import os, sys, json, subprocess, shutil, zipfile, hashlib, math, time
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("不在 Colab 里")
DRIVE = Path(DRIVE_DIR)
REPO = Path("/content") / REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS}", "onnxruntime", "pyyaml"], check=True)
sys.path.insert(0, str(REPO)); os.chdir(REPO)
print("repo commit:", subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
PRE = json.loads((REPO / "sequential" / "seq_prelock.json").read_text(encoding="utf-8"))
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""): h.update(b)
    return h.hexdigest()
if not (REPO / "data" / "merged_v5").exists():
    local = Path("/content") / "merged_v5.zip"
    if not local.exists(): shutil.copy2(DRIVE / "merged_v5.zip", local)
    with zipfile.ZipFile(local) as z: z.extractall(REPO / "data")
MERGED = REPO / "data" / "merged_v5"
VAL_I, VAL_L = MERGED / "images" / "val", MERGED / "labels" / "val"
onnx_path = DRIVE / ONNX_NAME
assert sha256(onnx_path) == PRE["v0"]["onnx"]["sha256"], "不是预锁定里的 v0"
from dlcore.metrics import EVALUATOR_VERSION
print("v0 哈希一致；评估器版本", EVALUATOR_VERSION)

In [ ]:
# 1) 标注和图片：我们读到的框数 vs Ultralytics（预锁定参考里的 n_instances），重复框、越界坐标、EXIF 旋转
import numpy as np, cv2
from PIL import Image, ImageOps
from dlcore.common import PRODUCT, images_in, read_yolo, xywhn_to_xyxy_px
from ultralytics.utils.patches import imread as ul_imread
imgs = images_in(VAL_I)
cnt = np.zeros(len(PRODUCT), int); dup = 0; bad_imgs = []; exif_rot = 0; shape_mismatch = []
for p in imgs:
    g = read_yolo(VAL_L / f"{p.stem}.txt"); g = np.zeros((0, 5)) if g is None else g
    for c in g[:, 0].astype(int): cnt[c] += 1
    if len(g):
        dup += len(g) - len(np.unique(g, axis=0))
        mn, mx = g[:, 1:].min(), g[:, 1:].max()
        if mx > 1.01 or mn < -0.01 or (mn < 0 and g[:, 3:5].min() < 0):
            bad_imgs.append(p.name)
    with Image.open(p) as im:
        o = im.getexif().get(0x0112, 1); s1 = ImageOps.exif_transpose(im).size
    exif_rot += o not in (None, 1)
    h, w = ul_imread(str(p)).shape[:2]
    if (w, h) != tuple(s1): shape_mismatch.append(p.name)
ref = PRE["v0"]["reference_val_onnx"]["per_class"]
print(f"val 图片 {len(imgs)} 张；EXIF 需要旋转的 {exif_rot} 张；PIL 和 OpenCV 读出来尺寸不同的 {len(shape_mismatch)} 张 {shape_mismatch[:5]}")
print(f"重复的框（完全相同的行）{dup} 个；Ultralytics 会当成坏图跳过的图 {len(bad_imgs)} 张 {bad_imgs[:5]}")
print("每类框数  我们读到 / Ultralytics（预锁定参考）")
for i, c in enumerate(PRODUCT):
    print(f"  {c:<20} {cnt[i]:>5} / {ref.get(c, {}).get('n_instances', 'n/a')}")
DIAG = {"n_images": len(imgs), "exif_rotated": int(exif_rot), "shape_mismatch": shape_mismatch, "duplicate_boxes": int(dup),
        "ultralytics_corrupt_images": bad_imgs, "gt_counts_ours": {c: int(cnt[i]) for i, c in enumerate(PRODUCT)}}

In [ ]:
# 2) 预测：三种预处理，各配两种标注读法
from dlcore.infer import Detector, load_image, letterbox, nms, MAX_WH, MAX_NMS
from dlcore.metrics import match_tp, ap_per_class
from ultralytics.data.augment import LetterBox
det = Detector(str(onnx_path))

def post(p, conf=0.001, iou=0.7, max_det=300):
    cs = p[:, 4:]; i, j = np.nonzero(cs > conf)
    s, c, xywh = cs[i, j], j, p[i, :4]
    if not len(s): return np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)
    if len(s) > MAX_NMS:
        k = np.argsort(-s, kind="stable")[:MAX_NMS]; s, c, xywh = s[k], c[k], xywh[k]
    b = np.concatenate([xywh[:, :2] - xywh[:, 2:] / 2, xywh[:, :2] + xywh[:, 2:] / 2], 1)
    keep = nms(b + c[:, None] * MAX_WH, s, iou, max_det)
    return b[keep].astype(float), s[keep].astype(float), c[keep].astype(int)

def prep_server(p):              # 本仓库 = 服务器：PIL 读图（EXIF 旋转）、PIL 双线性一次缩放到 640、灰边居中
    im = load_image(p); w0, h0 = im.size
    x, r, left, top = letterbox(im, 640)
    return x, (round(w0 * r), round(h0 * r), left, top)

def make_prep_ul(interp):        # Ultralytics val：OpenCV 读图、长边缩到 640（ceil）、LetterBox(scaleup=False)、BGR->RGB
    def prep(p):
        im = ul_imread(str(p)); h0, w0 = im.shape[:2]; r = 640 / max(h0, w0)
        if r != 1:
            im = cv2.resize(im, (min(math.ceil(w0 * r), 640), min(math.ceil(h0 * r), 640)), interpolation=interp)
        h1, w1 = im.shape[:2]
        lb = LetterBox(new_shape=(640, 640), scaleup=False)
        prm = lb.get_params({"img": im}); out = lb(image=im)
        x = np.ascontiguousarray(out[..., ::-1].transpose(2, 0, 1)[None], dtype=np.float32) / 255.0
        return x, (w1, h1, prm["left"], prm["top"])
    return prep

def predict_all(prep, name):
    t0 = time.time(); res = []
    for k, p in enumerate(imgs):
        x, geo = prep(p)
        raw = det.session.run(None, {det.input_name: x})[0][0].T
        res.append((geo, *post(raw)))
        if (k + 1) % 150 == 0: print(f"  {name}: {k + 1}/{len(imgs)}")
    print(f"  {name}: 完成（{time.time() - t0:.0f} s）")
    return res

def score(preds, ul_gt):
    tps, confs, pcs, gcs = [], [], [], []
    for p, ((w1, h1, left, top), b, s, c) in zip(imgs, preds):
        g = read_yolo(VAL_L / f"{p.stem}.txt"); g = np.zeros((0, 5)) if g is None else g
        if ul_gt:
            if p.name in bad_imgs: continue          # Ultralytics 跳过整张图
            if len(g): g = np.unique(g, axis=0)      # Ultralytics 去掉重复的框
        gb = xywhn_to_xyxy_px(g, w1, h1) + np.array([left, top, left, top])
        gc = g[:, 0].astype(int)
        tps.append(match_tp(gc, gb, c, b, 0.5, s)); confs.append(s); pcs.append(c); gcs.append(gc)
    ap = ap_per_class(np.concatenate(tps), np.concatenate(confs), np.concatenate(pcs), np.concatenate(gcs))
    return float(np.mean(list(ap.values()))), {PRODUCT[k]: v for k, v in ap.items()}

variants = {"A 服务器预处理（PIL）": prep_server,
            "B Ultralytics 预处理（OpenCV INTER_LINEAR）": make_prep_ul(cv2.INTER_LINEAR),
            "C OpenCV INTER_AREA（抗锯齿）": make_prep_ul(cv2.INTER_AREA)}
PREDS = {n: predict_all(f, n) for n, f in variants.items()}

In [ ]:
# 3) 汇总（把这一格的输出整段贴给 Claude）
ul_ref = PRE["v0"]["reference_val_onnx"]["mAP50_all11"]
rows = {}
print(f"Ultralytics 8.4 val（预锁定参考）mAP50 = {ul_ref:.4f}\n")
for n, pr in PREDS.items():
    for ul_gt in (False, True):
        m, ap = score(pr, ul_gt)
        key = f"{n} + {'Ultralytics 标注规则' if ul_gt else '本仓库标注规则'}"
        rows[key] = {"mAP50": m, "diff_vs_ultralytics": m - ul_ref, "AP50": ap}
        print(f"{key:<60} mAP50 {m:.4f}   差 {m - ul_ref:+.4f}")
print("\n每类 AP50（B + Ultralytics 标注规则 vs 预锁定参考）")
bk = "B Ultralytics 预处理（OpenCV INTER_LINEAR） + Ultralytics 标注规则"
for c in PRODUCT:
    r = PRE["v0"]["reference_val_onnx"]["per_class"].get(c, {}).get("ap50")
    v = rows[bk]["AP50"].get(c)
    print(f"  {c:<20} 诊断 {v if v is None else round(v, 4)}   参考 {r if r is None else round(r, 4)}")
DIAG["variants"] = rows; DIAG["ultralytics_reference_mAP50"] = ul_ref
(DRIVE / "evaluator_diagnosis.json").write_text(json.dumps(DIAG, indent=2, ensure_ascii=False))
print("\n已存到 Drive：", DRIVE / "evaluator_diagnosis.json")